# Example Temporal Logic Monitoring in Python

mstlo is a Python library for Online Monitoring of Signal Temporal Logic (STL) specifications. Its design is inherently online, but wrappers allow for offline monitoring as well. This tutorial demonstrates how to use mstlo for monitoring STL specifications in an online settings.

## Installation

Install mstlo as a Python package:

In [18]:
%pip install mstlo-python

Note: you may need to restart the kernel to use updated packages.


## Usage

### Monitoring STL Specifications
To perform monitoring of STL specifications using mstlo, use either the `update_batch` method to process a batch of signal data or the `update` method to process one signal value at a time. The `update_batch` method takes a dictionary of signal names to lists of (timestamp, value) pairs and the `update` method takes three arguments: the signal name, the timestamp, and the signal value. Both methods return a `MonitorOutput` object, which contains the robustness values for the monitored specification at the given time.

The monitor will keep track of the signal history and output the robustness values accordingly.

### Building a monitor
Below shows an example of how to build a monitor. 

In [19]:
import mstlo_python as mstlo

spec = 'globally[0,2] (x >= 3) -> eventually[0,5] (y >= 3)'
spec = mstlo.parse_formula(spec)

monitor = mstlo.Monitor(formula=spec, semantics="Rosi")
print(monitor)

STL Monitor Configuration:
  Specification: (¬(G[0, 2](¬(x < 3)))) v (F[0, 5](¬(y < 3)))
  Algorithm: Incremental
  Semantics: RobustnessInterval
  Synchronization: ZeroOrderHold
  Temporal Depth: 5s



Let us break down the monitor configuration above:

- **Specification:** The one provided. Note that the specification seen by the monitor is slightly different from the one we wrote. This is due to mstlo automatically rewriting the specification to a form that the monitor can process (known as 'syntactic sugar') using only the base STL syntax. For instance, the 'implies' operator employs the logical law: $p \rightarrow q \equiv \neg p \lor q$.
- **Algorithm:** mstlo uses a *incremental* algorithm which is far more efficient for online monitoring. The alternative is the *Naive* algorithm, which is less efficient.
- **Semantics:** RobustnessInterval (a.k.a. RoSI [[Deshmukh et al 2015]](https://arxiv.org/abs/1506.08234)). This is the standard online semantics for STL monitoring, outputting a robustness interval for each step. Other supported semantics are:
  - Delayed quantitative semantics: outputs a single robustness value for each step when the temporal depth of the specification is exceeded.
  - Delayed qualitative semantics: outputs a boolean value for each step when the temporal depth of the specification is exceeded, which is True if the specification is satisfied and False if it is violated.
  - Eager qualitative semantics: outputs a boolean value for each step, but also outputs an early verdict when possible. 
- **Synchronization:** Set to ZeroOrderHold, which means that the monitor will use the most recent value of the signal for each timestamp. This is the default synchronization method. Linear is also supported, which linearly interpolates between signal values. **Note:** If only one signal is involved in the specification, synchronization is not needed and will be set to None automatically.
- **Temporal depth:** Computed to 5s ($\max(5,2)$). This means that the monitor can output a certain robustness value after seeing 5s of the signal. This is due to the temporal operators in the specification. The temporal depth is the maximum time horizon that the monitor needs to keep track of in order to evaluate the specification.

See also the examples at https://github.com/INTO-CPS-Association/mstlo/tree/main/mstlo-python/examples for more monitor configurations.

### Updating the monitor with signal data
Now let us feed the monitor with the signal data. The `update_batch` method can be called multiple times, and the monitor will keep track of the signal history and output the robustness values accordingly.

In [20]:
x_steps = [(6,0), (10, 4), (5,8), (3, 12)] # value, time
y_steps = [(0,0), (6, 4), (0,8), (9, 12)] # value, time

for x, y in zip(x_steps, y_steps):
    verdict = monitor.update_batch({'x': [x], 'y': [y]})
    # verdict_list = verdict.verdicts() # uncomment this line to get the list of verdicts for each step
    print(f"Feeding x={x[0]} at t={x[1]} and y={y[0]} at t={y[1]} \n verdict=\n{verdict}")
    print("-"*50)

Feeding x=6 at t=0 and y=0 at t=0 
 verdict=
t=0ns: RobustnessInterval(-3.0, inf)
--------------------------------------------------
Feeding x=10 at t=4 and y=6 at t=4 
 verdict=
t=0ns: RobustnessInterval(3.0, inf)
t=4s: RobustnessInterval(3.0, inf)
--------------------------------------------------
Feeding x=5 at t=8 and y=0 at t=8 
 verdict=
t=0ns: RobustnessInterval(3.0, 3.0)
t=4s: RobustnessInterval(3.0, inf)
t=8s: RobustnessInterval(-2.0, inf)
--------------------------------------------------
Feeding x=3 at t=12 and y=9 at t=12 
 verdict=
t=4s: RobustnessInterval(3.0, 3.0)
t=8s: RobustnessInterval(6.0, inf)
t=12s: RobustnessInterval(6.0, inf)
--------------------------------------------------


In the above, a verdict is outputted for each timestamp in the signal data, until the temporal depth of the specification is reached after which the verdict is finalized (i.e. upper and lower bounds of the robustness interval are equal). Note that these semantics allow for early verdicts in the sense that a positive lower bound means that the specification is satisfied regardless of future signal values, and a negative upper bound means that the specification is violated regardless of future signal values. This is why the monitor outputs a robustness interval for each timestamp, as it captures the uncertainty in the verdict until the temporal depth is reached.

## Specifications with Parameters
mstlo also supports specifications with parameters, which can be updated at runtime. This allows for more flexible monitoring, as the same specification can be used with different parameter values without having to recompile the specification. To use parameters, simply define them in the specification using the `$` syntax, and then set their values in the `Variables` object passed to the monitor. The monitor will then use these parameter values when evaluating the specification.

The example below illustrates this. 

In [21]:
phi = mstlo.parse_formula('G[0,100] (T < $max_T)')
vars = mstlo.Variables()
vars.set('max_T', 50)
monitor = mstlo.Monitor(formula=phi, semantics="Rosi", variables=vars)
print(monitor)

STL Monitor Configuration:
  Specification: G[0, 100](T < $max_T)
  Algorithm: Incremental
  Semantics: RobustnessInterval
  Synchronization: None
  Temporal Depth: 100s
  Variables:
    $max_T = 50



In [22]:
assert(monitor.get_variables().get('max_T') == 50)
out = monitor.update('T', 100, 0)
print(out)

t=0ns: RobustnessInterval(-inf, -50.0)


In [23]:
vars.set('max_T', 120)
assert(monitor.get_variables().get('max_T') == 120)
out = monitor.update('T', 100, 10)
print(out)

t=0ns: RobustnessInterval(-inf, -50.0)
t=10s: RobustnessInterval(-inf, 20.0)


## Exercises

### 1: Alternative semantics
Try changing the semantics of the monitor to the other supported semantics and see how the output changes. What are the differences between the different semantics? Any trade-offs to consider when choosing one over the other?
